# ANSC 4040 Mini Project  
## Extra Trees Model Development for Missing `AnimalId` Prediction

This notebook is an updated version of the existing `Model_building.ipynb` workflow.

It keeps the chronological **60% training / 20% validation / 20% test** design and develops an `ExtraTreesClassifier` workflow for multiclass `AnimalId` prediction.

### Important model-structure note

Extra Trees is **not a neural network**, so it does **not** contain hidden layers or a conventional output layer.

Its internal model structure consists of many randomized decision trees:

- `n_estimators`: number of trees
- `max_depth`: maximum depth of each tree
- `min_samples_leaf`: minimum observations in a terminal leaf
- `max_features`: number of predictors considered at each split

The model output is a probability distribution across the `AnimalId` classes. The class with the highest probability is used as the predicted `AnimalId`.

### Workflow

1. Load the modelling dataset.
2. Check missing values and target structure.
3. Sort observations chronologically by `EventDate`.
4. Split labelled observations into 60% train, 20% validation, and 20% test.
5. Build date and categorical features.
6. Encode `AnimalId` using training-period classes.
7. Measure validation/test class coverage.
8. Use class-aware sampling for computationally practical model development.
9. Tune Extra Trees hyperparameters using the validation period only.
10. Refit the selected configuration on train + validation.
11. Evaluate once on the held-out test period.
12. Refit a production model using all known `AnimalId` records.
13. Predict missing `AnimalId` records in batches and save prediction confidence.


In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    top_k_accuracy_score,
)

RANDOM_STATE = 42

# Update this path if the modelling CSV is stored elsewhere.
MODEL_DATA_PATH = Path(
    r"D:\ansci-4040-fall-2026\jl4937_6040-project-1\dataset\dataset_for_model_buliding_2.csv"
)

# Development sampling:
# keeps at most this many records per AnimalId.
MAX_ROWS_PER_ANIMAL_TUNING = 300

# Validation sample used repeatedly during hyperparameter tuning.
MAX_VALIDATION_ROWS_TUNING = 100_000

# Final train+validation fit.
MAX_ROWS_PER_ANIMAL_FINAL = 400

# Production fit on all labelled records.
MAX_ROWS_PER_ANIMAL_PRODUCTION = 400

# Batch size for final missing-ID prediction.
PREDICTION_BATCH_SIZE = 50_000


In [ ]:
# ============================================================
# 2. Load modelling data
# ============================================================

data = pd.read_csv(
    MODEL_DATA_PATH,
    low_memory=False
)

print(f"Rows: {len(data):,}")
print(f"Columns: {data.shape[1]:,}")
display(data.head())


In [ ]:
# ============================================================
# 3. Basic data checks
# ============================================================

# Count null values by column.
null_counts = data.isna().sum()

# Count zero values in numeric columns.
zero_counts = (
    data.select_dtypes(include="number")
        .eq(0)
        .sum()
)

summary = pd.DataFrame({
    "null_count": null_counts,
    "zero_count": zero_counts.reindex(
        data.columns,
        fill_value=0
    )
})

display(summary)

print(
    f"\nTotal null values: "
    f"{data.isna().sum().sum():,}"
)

print(
    f"Total zero values: "
    f"{zero_counts.sum():,}"
)

animal_id_count = data["AnimalId"].nunique(dropna=True)

print(
    f"\nUnique known AnimalId values: "
    f"{animal_id_count:,}"
)

print(
    f"AnimalId uniqueness ratio: "
    f"{animal_id_count / len(data):.2%}"
)

print(
    f"Missing AnimalId rows in this modelling file: "
    f"{data['AnimalId'].isna().sum():,}"
)


## Chronological split

The modelling data are sorted from the earliest to latest `EventDate` and split without shuffling. This preserves the temporal structure of the records and reduces temporal leakage.

Only records with a **known** `AnimalId` are used for train/validation/test evaluation. Records with missing `AnimalId` belong to the later prediction/inference stage.


In [ ]:
# ============================================================
# 4. Prepare labelled data and sort chronologically
# ============================================================

data["EventDate"] = pd.to_datetime(
    data["EventDate"],
    errors="coerce"
)

# Keep labelled records for model development/evaluation.
labelled_data = (
    data.loc[data["AnimalId"].notna()]
        .copy()
)

# Remove records without a usable date because the split is temporal.
missing_date_rows = labelled_data["EventDate"].isna().sum()

if missing_date_rows > 0:
    print(
        f"Removing {missing_date_rows:,} labelled rows "
        f"with missing/unparseable EventDate."
    )
    labelled_data = labelled_data.loc[
        labelled_data["EventDate"].notna()
    ].copy()

labelled_data = (
    labelled_data
    .sort_values(
        "EventDate",
        ascending=True
    )
    .reset_index(drop=True)
)

print(labelled_data[["EventDate"]].head())
print(labelled_data[["EventDate"]].tail())


In [ ]:
# ============================================================
# 5. Chronological 60/20/20 train/validation/test split
# ============================================================

n = len(labelled_data)

train_end = int(n * 0.60)
validation_end = train_end + int(n * 0.20)

data_train = (
    labelled_data.iloc[:train_end]
    .copy()
)

data_validation = (
    labelled_data.iloc[
        train_end:validation_end
    ]
    .copy()
)

data_test = (
    labelled_data.iloc[
        validation_end:
    ]
    .copy()
)

print(
    f"Training set:   "
    f"{len(data_train):,} rows "
    f"({len(data_train) / n:.1%})"
)

print(
    f"Validation set: "
    f"{len(data_validation):,} rows "
    f"({len(data_validation) / n:.1%})"
)

print(
    f"Test set:       "
    f"{len(data_test):,} rows "
    f"({len(data_test) / n:.1%})"
)

print("\nDate ranges:")

print(
    f"Training:   "
    f"{data_train['EventDate'].min()} "
    f"to {data_train['EventDate'].max()}"
)

print(
    f"Validation: "
    f"{data_validation['EventDate'].min()} "
    f"to {data_validation['EventDate'].max()}"
)

print(
    f"Test:       "
    f"{data_test['EventDate'].min()} "
    f"to {data_test['EventDate'].max()}"
)


## Feature engineering

`AnimalId` is removed from the predictors. `EventDate` is converted into time-based predictors instead of being passed directly to Extra Trees.

The same training-period origin date is used for train, validation, and test to avoid inconsistent date transformations.


In [ ]:
# ============================================================
# 6. Feature engineering functions
# ============================================================

def make_features(frame, origin_date):
    """Create model-ready predictors without AnimalId."""

    features = frame.drop(
        columns=["AnimalId"],
        errors="ignore"
    ).copy()

    features["EventDate"] = pd.to_datetime(
        features["EventDate"],
        errors="coerce"
    )

    features["EventYear"] = (
        features["EventDate"].dt.year
    )

    features["EventMonth"] = (
        features["EventDate"].dt.month
    )

    features["EventDay"] = (
        features["EventDate"].dt.day
    )

    features["EventDayOfWeek"] = (
        features["EventDate"].dt.dayofweek
    )

    features["EventDayOfYear"] = (
        features["EventDate"].dt.dayofyear
    )

    features["DaysFromStart"] = (
        features["EventDate"] - origin_date
    ).dt.days

    features = features.drop(
        columns=["EventDate"]
    )

    return features


def one_hot_fit_transform(train_features):
    """One-hot encode categorical columns using training data."""

    categorical_cols = (
        train_features
        .select_dtypes(
            include=["object", "category"]
        )
        .columns
        .tolist()
    )

    encoded = pd.get_dummies(
        train_features,
        columns=categorical_cols,
        dtype=np.uint8
    )

    return encoded, categorical_cols


def one_hot_transform(
    features,
    categorical_cols,
    reference_columns
):
    """Apply dummy encoding and align columns to the training matrix."""

    existing_categorical_cols = [
        col
        for col in categorical_cols
        if col in features.columns
    ]

    encoded = pd.get_dummies(
        features,
        columns=existing_categorical_cols,
        dtype=np.uint8
    )

    encoded = encoded.reindex(
        columns=reference_columns,
        fill_value=0
    )

    return encoded


def downcast_numeric(frame):
    """Reduce memory use for large model matrices."""

    result = frame.copy()

    float_cols = result.select_dtypes(
        include=["float64"]
    ).columns

    int_cols = result.select_dtypes(
        include=["int64"]
    ).columns

    result[float_cols] = result[
        float_cols
    ].astype(np.float32)

    result[int_cols] = result[
        int_cols
    ].apply(
        pd.to_numeric,
        downcast="integer"
    )

    return result


In [ ]:
# ============================================================
# 7. Create and align train/validation/test feature matrices
# ============================================================

origin_date = data_train["EventDate"].min()

X_train_raw = make_features(
    data_train,
    origin_date
)

X_validation_raw = make_features(
    data_validation,
    origin_date
)

X_test_raw = make_features(
    data_test,
    origin_date
)

X_train, categorical_cols = (
    one_hot_fit_transform(
        X_train_raw
    )
)

X_validation = one_hot_transform(
    X_validation_raw,
    categorical_cols,
    X_train.columns
)

X_test = one_hot_transform(
    X_test_raw,
    categorical_cols,
    X_train.columns
)

X_train = downcast_numeric(X_train)
X_validation = downcast_numeric(X_validation)
X_test = downcast_numeric(X_test)

print("Categorical variables:")
print(categorical_cols)

print("\nFeature matrix shapes:")
print("Train:      ", X_train.shape)
print("Validation: ", X_validation.shape)
print("Test:       ", X_test.shape)


## Target encoding and class coverage

A standard multiclass classifier can predict only `AnimalId` classes it has already seen during training.

Therefore, validation/test records whose `AnimalId` is absent from the earlier training period are not ordinary classification errors. The notebook reports **coverage** separately:

- **Validation coverage** = proportion of validation rows whose `AnimalId` occurred during training.
- **Test coverage** = proportion of test rows whose `AnimalId` occurred during the relevant training period.

Accuracy and F1 metrics are calculated only on evaluable (seen-class) records.


In [ ]:
# ============================================================
# 8. Encode AnimalId from TRAINING classes only
# ============================================================

train_ids, class_values = pd.factorize(
    data_train["AnimalId"]
)

class_to_number = {
    animal_id: class_number
    for class_number, animal_id
    in enumerate(class_values)
}

number_to_class = {
    class_number: animal_id
    for class_number, animal_id
    in enumerate(class_values)
}

y_train = pd.Series(
    train_ids,
    index=data_train.index,
    dtype=np.int32
)

validation_ids = (
    data_validation["AnimalId"]
    .map(class_to_number)
)

test_ids = (
    data_test["AnimalId"]
    .map(class_to_number)
)

known_validation_mask = (
    validation_ids.notna()
)

known_test_mask = (
    test_ids.notna()
)

validation_coverage = (
    known_validation_mask.mean()
)

test_coverage_from_train_only = (
    known_test_mask.mean()
)

print(
    f"Training AnimalId classes: "
    f"{len(class_values):,}"
)

print(
    f"Validation row coverage from training classes: "
    f"{validation_coverage:.2%}"
)

print(
    f"Test row coverage from training classes: "
    f"{test_coverage_from_train_only:.2%}"
)

X_validation_known = (
    X_validation.loc[
        known_validation_mask
    ]
    .copy()
)

y_validation = (
    validation_ids.loc[
        known_validation_mask
    ]
    .astype(np.int32)
)

X_test_known_from_train = (
    X_test.loc[
        known_test_mask
    ]
    .copy()
)

y_test_from_train = (
    test_ids.loc[
        known_test_mask
    ]
    .astype(np.int32)
)


## Class-aware sampling

The dataset contains millions of records. Instead of drawing a purely random global sample, the tuning sample retains up to a fixed number of records **per `AnimalId`**. This reduces the chance that lower-frequency animals disappear from the training sample.


In [ ]:
# ============================================================
# 9. Class-aware sampling helper
# ============================================================

def sample_per_class(
    frame,
    target_col="AnimalId",
    max_rows_per_class=300,
    random_state=42
):
    """Return row indices with at most max_rows_per_class for each target class."""

    sampled_indices = []

    for _, group in frame.groupby(
        target_col,
        sort=False
    ):
        n_sample = min(
            len(group),
            max_rows_per_class
        )

        sampled = group.sample(
            n=n_sample,
            replace=False,
            random_state=random_state
        )

        sampled_indices.extend(
            sampled.index.tolist()
        )

    return pd.Index(sampled_indices)


train_sample_index = sample_per_class(
    data_train,
    max_rows_per_class=MAX_ROWS_PER_ANIMAL_TUNING,
    random_state=RANDOM_STATE
)

X_train_sample = (
    X_train.loc[
        train_sample_index
    ]
)

y_train_sample = (
    y_train.loc[
        train_sample_index
    ]
)

print(
    f"Full training rows: "
    f"{len(X_train):,}"
)

print(
    f"Sampled training rows: "
    f"{len(X_train_sample):,}"
)

print(
    f"AnimalId classes in full training: "
    f"{y_train.nunique():,}"
)

print(
    f"AnimalId classes in sampled training: "
    f"{y_train_sample.nunique():,}"
)

if y_train_sample.nunique() != y_train.nunique():
    raise ValueError(
        "The class-aware sample unexpectedly lost one or more AnimalId classes."
    )


In [ ]:
# ============================================================
# 10. Validation sample for repeated hyperparameter evaluation
# ============================================================

rng = np.random.default_rng(
    RANDOM_STATE
)

max_validation_rows = min(
    MAX_VALIDATION_ROWS_TUNING,
    len(X_validation_known)
)

validation_positions = rng.choice(
    len(X_validation_known),
    size=max_validation_rows,
    replace=False
)

X_validation_sample = (
    X_validation_known.iloc[
        validation_positions
    ]
)

y_validation_sample = (
    y_validation.iloc[
        validation_positions
    ]
)

print(
    f"Validation rows used during tuning: "
    f"{len(X_validation_sample):,}"
)


## Extra Trees hyperparameter optimization

This is a real validation-based hyperparameter search. Each configuration changes the internal tree structure.

The notebook selects the best configuration using **Top-1 validation accuracy**, because the final task requires one predicted `AnimalId`. Top-5 accuracy is also reported as a secondary diagnostic.


In [ ]:
# ============================================================
# 11. Hyperparameter configurations
# ============================================================

parameter_grid = [
    {
        "n_estimators": 150,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": 25,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": 30,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 300,
        "max_depth": 30,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
]

print(
    f"Number of Extra Trees configurations: "
    f"{len(parameter_grid)}"
)


In [ ]:
# ============================================================
# 12. Extra Trees hyperparameter tuning loop
# ============================================================

tuning_results = []

best_model = None
best_score = -np.inf
best_parameters = None

for configuration_number, params in enumerate(
    parameter_grid,
    start=1
):
    print("\n" + "=" * 70)

    print(
        f"Configuration "
        f"{configuration_number} / "
        f"{len(parameter_grid)}"
    )

    print(params)
    print("=" * 70)

    model = ExtraTreesClassifier(
        n_estimators=params[
            "n_estimators"
        ],
        max_depth=params[
            "max_depth"
        ],
        min_samples_leaf=params[
            "min_samples_leaf"
        ],
        max_features=params[
            "max_features"
        ],
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    print("Training model...")

    model.fit(
        X_train_sample,
        y_train_sample
    )

    validation_predictions = model.predict(
        X_validation_sample
    )

    accuracy = accuracy_score(
        y_validation_sample,
        validation_predictions
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_validation_sample,
            validation_predictions
        )
    )

    macro_f1 = f1_score(
        y_validation_sample,
        validation_predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_validation_sample,
        validation_predictions,
        average="weighted",
        zero_division=0
    )

    probabilities = model.predict_proba(
        X_validation_sample
    )

    model_classes = model.classes_

    top5_accuracy = top_k_accuracy_score(
        y_validation_sample,
        probabilities,
        k=min(
            5,
            len(model_classes)
        ),
        labels=model_classes
    )

    result = {
        "configuration": configuration_number,
        "n_estimators": params[
            "n_estimators"
        ],
        "max_depth": params[
            "max_depth"
        ],
        "min_samples_leaf": params[
            "min_samples_leaf"
        ],
        "max_features": params[
            "max_features"
        ],
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "top5_accuracy": top5_accuracy,
    }

    tuning_results.append(result)

    # Primary model-selection metric:
    # exact AnimalId classification accuracy.
    selection_score = accuracy

    if selection_score > best_score:
        best_score = selection_score
        best_model = model
        best_parameters = params.copy()

    print(
        f"Accuracy:          "
        f"{accuracy:.4f}"
    )

    print(
        f"Balanced accuracy: "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"Macro F1:          "
        f"{macro_f1:.4f}"
    )

    print(
        f"Weighted F1:       "
        f"{weighted_f1:.4f}"
    )

    print(
        f"Top-5 accuracy:    "
        f"{top5_accuracy:.4f}"
    )


In [ ]:
# ============================================================
# 13. Compare tuning results
# ============================================================

tuning_results_df = (
    pd.DataFrame(
        tuning_results
    )
    .sort_values(
        "accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

display(tuning_results_df)

print("\nBest parameters:")
print(best_parameters)

print(
    f"\nBest sampled-validation accuracy: "
    f"{best_score:.4f}"
)


In [ ]:
# ============================================================
# 14. Evaluate selected configuration on FULL validation period
# ============================================================

validation_predictions_full = (
    best_model.predict(
        X_validation_known
    )
)

validation_accuracy_full = (
    accuracy_score(
        y_validation,
        validation_predictions_full
    )
)

validation_balanced_accuracy_full = (
    balanced_accuracy_score(
        y_validation,
        validation_predictions_full
    )
)

validation_macro_f1_full = (
    f1_score(
        y_validation,
        validation_predictions_full,
        average="macro",
        zero_division=0
    )
)

validation_weighted_f1_full = (
    f1_score(
        y_validation,
        validation_predictions_full,
        average="weighted",
        zero_division=0
    )
)

print(
    f"Validation coverage: "
    f"{validation_coverage:.2%}"
)

print(
    f"Full validation accuracy: "
    f"{validation_accuracy_full:.4f}"
)

print(
    f"Full validation balanced accuracy: "
    f"{validation_balanced_accuracy_full:.4f}"
)

print(
    f"Full validation macro F1: "
    f"{validation_macro_f1_full:.4f}"
)

print(
    f"Full validation weighted F1: "
    f"{validation_weighted_f1_full:.4f}"
)


In [ ]:
# ============================================================
# 15. Feature importance from selected validation model
# ============================================================

feature_importance = (
    pd.DataFrame({
        "Feature": X_train_sample.columns,
        "Importance": best_model.feature_importances_,
    })
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    feature_importance.head(20)
)

top_features = feature_importance.head(15)

plt.figure(
    figsize=(9, 6)
)

plt.barh(
    top_features["Feature"][::-1],
    top_features["Importance"][::-1]
)

plt.xlabel(
    "Feature Importance"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Top 15 Feature Importances - Extra Trees"
)

plt.tight_layout()
plt.show()


# Final held-out test evaluation

After hyperparameters have been selected from the validation period, the model is refitted on **train + validation** and evaluated once on the test period.

The test set should not be used to further tune `n_estimators`, `max_depth`, `min_samples_leaf`, or `max_features`.


In [ ]:
# ============================================================
# 16. Combine train + validation for final test model
# ============================================================

data_train_final = pd.concat(
    [
        data_train,
        data_validation,
    ],
    ignore_index=True
)

origin_date_final = (
    data_train_final["EventDate"].min()
)

X_train_final_raw = make_features(
    data_train_final,
    origin_date_final
)

X_test_final_raw = make_features(
    data_test,
    origin_date_final
)

X_train_final, categorical_cols_final = (
    one_hot_fit_transform(
        X_train_final_raw
    )
)

X_test_final = one_hot_transform(
    X_test_final_raw,
    categorical_cols_final,
    X_train_final.columns
)

X_train_final = downcast_numeric(
    X_train_final
)

X_test_final = downcast_numeric(
    X_test_final
)

print(
    "Final train feature shape:",
    X_train_final.shape
)

print(
    "Test feature shape:",
    X_test_final.shape
)


In [ ]:
# ============================================================
# 17. Re-encode AnimalId using train + validation classes
# ============================================================

final_train_ids, final_class_values = (
    pd.factorize(
        data_train_final[
            "AnimalId"
        ]
    )
)

final_class_to_number = {
    animal_id: class_number
    for class_number, animal_id
    in enumerate(
        final_class_values
    )
}

final_number_to_class = {
    class_number: animal_id
    for class_number, animal_id
    in enumerate(
        final_class_values
    )
}

y_train_final = pd.Series(
    final_train_ids,
    index=data_train_final.index,
    dtype=np.int32
)

test_ids_final = (
    data_test["AnimalId"]
    .map(
        final_class_to_number
    )
)

known_test_final = (
    test_ids_final.notna()
)

final_test_coverage = (
    known_test_final.mean()
)

X_test_evaluable = (
    X_test_final.loc[
        known_test_final
    ]
)

y_test_final = (
    test_ids_final.loc[
        known_test_final
    ]
    .astype(np.int32)
)

print(
    f"Final test coverage after training on "
    f"train + validation: "
    f"{final_test_coverage:.2%}"
)


In [ ]:
# ============================================================
# 18. Class-aware sampling for final train + validation fit
# ============================================================

final_sample_index = sample_per_class(
    data_train_final,
    max_rows_per_class=MAX_ROWS_PER_ANIMAL_FINAL,
    random_state=RANDOM_STATE
)

X_train_final_sample = (
    X_train_final.loc[
        final_sample_index
    ]
)

y_train_final_sample = (
    y_train_final.loc[
        final_sample_index
    ]
)

print(
    f"Train + validation rows available: "
    f"{len(X_train_final):,}"
)

print(
    f"Rows used for final Extra Trees fit: "
    f"{len(X_train_final_sample):,}"
)

print(
    f"AnimalId classes retained: "
    f"{y_train_final_sample.nunique():,} / "
    f"{y_train_final.nunique():,}"
)


In [ ]:
# ============================================================
# 19. Fit final Extra Trees model using selected hyperparameters
# ============================================================

final_model = ExtraTreesClassifier(
    n_estimators=best_parameters[
        "n_estimators"
    ],
    max_depth=best_parameters[
        "max_depth"
    ],
    min_samples_leaf=best_parameters[
        "min_samples_leaf"
    ],
    max_features=best_parameters[
        "max_features"
    ],
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

print(
    "Training final Extra Trees model..."
)

final_model.fit(
    X_train_final_sample,
    y_train_final_sample
)

print(
    "Final model training completed."
)


In [ ]:
# ============================================================
# 20. ONE-TIME held-out test evaluation
# ============================================================

test_predictions = (
    final_model.predict(
        X_test_evaluable
    )
)

test_accuracy = accuracy_score(
    y_test_final,
    test_predictions
)

test_balanced_accuracy = (
    balanced_accuracy_score(
        y_test_final,
        test_predictions
    )
)

test_macro_f1 = f1_score(
    y_test_final,
    test_predictions,
    average="macro",
    zero_division=0
)

test_weighted_f1 = f1_score(
    y_test_final,
    test_predictions,
    average="weighted",
    zero_division=0
)

test_probabilities = (
    final_model.predict_proba(
        X_test_evaluable
    )
)

test_top5_accuracy = (
    top_k_accuracy_score(
        y_test_final,
        test_probabilities,
        k=min(
            5,
            len(final_model.classes_)
        ),
        labels=final_model.classes_
    )
)

print("\nFINAL TEST RESULTS")

print(
    f"Coverage:          "
    f"{final_test_coverage:.2%}"
)

print(
    f"Accuracy:          "
    f"{test_accuracy:.4f}"
)

print(
    f"Balanced accuracy: "
    f"{test_balanced_accuracy:.4f}"
)

print(
    f"Macro F1:          "
    f"{test_macro_f1:.4f}"
)

print(
    f"Weighted F1:       "
    f"{test_weighted_f1:.4f}"
)

print(
    f"Top-5 accuracy:    "
    f"{test_top5_accuracy:.4f}"
)


# Production model for missing `AnimalId` prediction

After the held-out test evaluation is finished, all labelled records can be used to fit the production model.

The prediction dataset is a **separate inference dataset** containing records with missing `AnimalId`.

Because `predict_proba()` can require a large amount of memory when there are many `AnimalId` classes, predictions are generated in batches and only the maximum class probability (confidence) is retained.


In [ ]:
# ============================================================
# 21. Configure the missing-AnimalId prediction file
# ============================================================

# Change this to the actual file containing the rows whose
# AnimalId values are missing.
PREDICTION_DATA_PATH = Path(
    r"D:\ansci-4040-fall-2026\jl4937_6040-project-1\dataset\dataset_for_prediction.csv"
)

# Keep False until PREDICTION_DATA_PATH points to the correct file.
RUN_FINAL_PREDICTION = False

# Exploratory confidence threshold.
# This should later be calibrated using validation results.
CONFIDENCE_THRESHOLD = 0.80


In [ ]:
# ============================================================
# 22. Fit production Extra Trees model on all known AnimalIds
# ============================================================

known_all = (
    data.loc[
        data["AnimalId"].notna()
    ]
    .copy()
)

known_all["EventDate"] = pd.to_datetime(
    known_all["EventDate"],
    errors="coerce"
)

known_all = (
    known_all.loc[
        known_all["EventDate"].notna()
    ]
    .sort_values(
        "EventDate"
    )
    .reset_index(drop=True)
)

production_origin_date = (
    known_all["EventDate"].min()
)

X_production_raw = make_features(
    known_all,
    production_origin_date
)

X_production, production_categorical_cols = (
    one_hot_fit_transform(
        X_production_raw
    )
)

X_production = downcast_numeric(
    X_production
)

production_ids, production_class_values = (
    pd.factorize(
        known_all["AnimalId"]
    )
)

production_number_to_class = {
    class_number: animal_id
    for class_number, animal_id
    in enumerate(
        production_class_values
    )
}

y_production = pd.Series(
    production_ids,
    index=known_all.index,
    dtype=np.int32
)

production_sample_index = sample_per_class(
    known_all,
    max_rows_per_class=MAX_ROWS_PER_ANIMAL_PRODUCTION,
    random_state=RANDOM_STATE
)

X_production_sample = (
    X_production.loc[
        production_sample_index
    ]
)

y_production_sample = (
    y_production.loc[
        production_sample_index
    ]
)

production_model = ExtraTreesClassifier(
    n_estimators=best_parameters[
        "n_estimators"
    ],
    max_depth=best_parameters[
        "max_depth"
    ],
    min_samples_leaf=best_parameters[
        "min_samples_leaf"
    ],
    max_features=best_parameters[
        "max_features"
    ],
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

print(
    f"Training production model on "
    f"{len(X_production_sample):,} sampled labelled records..."
)

production_model.fit(
    X_production_sample,
    y_production_sample
)

print(
    "Production model training completed."
)


In [ ]:
# ============================================================
# 23. Batch prediction helper
# ============================================================

def predict_in_batches(
    model,
    X,
    number_to_class,
    batch_size=50_000
):
    """Predict classes and maximum class probability in memory-safe batches."""

    predicted_ids = []
    confidences = []

    for start in range(
        0,
        len(X),
        batch_size
    ):
        stop = min(
            start + batch_size,
            len(X)
        )

        X_batch = X.iloc[
            start:stop
        ]

        class_numbers = model.predict(
            X_batch
        )

        probabilities = model.predict_proba(
            X_batch
        )

        max_probability = probabilities.max(
            axis=1
        )

        decoded_ids = [
            number_to_class[int(class_number)]
            for class_number
            in class_numbers
        ]

        predicted_ids.extend(
            decoded_ids
        )

        confidences.extend(
            max_probability
        )

        print(
            f"Predicted rows "
            f"{start:,} to {stop:,}"
        )

    return (
        np.asarray(predicted_ids),
        np.asarray(confidences)
    )


In [ ]:
# ============================================================
# 24. Predict missing AnimalId records
# ============================================================

if RUN_FINAL_PREDICTION:

    prediction_data = pd.read_csv(
        PREDICTION_DATA_PATH,
        low_memory=False
    )

    prediction_data["EventDate"] = pd.to_datetime(
        prediction_data["EventDate"],
        errors="coerce"
    )

    missing_animal_data = (
        prediction_data.loc[
            prediction_data[
                "AnimalId"
            ].isna()
        ]
        .copy()
    )

    print(
        f"Rows requiring AnimalId prediction: "
        f"{len(missing_animal_data):,}"
    )

    if len(missing_animal_data) == 0:
        raise ValueError(
            "The prediction file contains no rows with missing AnimalId."
        )

    X_missing_raw = make_features(
        missing_animal_data,
        production_origin_date
    )

    X_missing = one_hot_transform(
        X_missing_raw,
        production_categorical_cols,
        X_production.columns
    )

    X_missing = downcast_numeric(
        X_missing
    )

    predicted_animal_ids, prediction_confidence = (
        predict_in_batches(
            production_model,
            X_missing,
            production_number_to_class,
            batch_size=PREDICTION_BATCH_SIZE
        )
    )

    missing_animal_data[
        "PredictedAnimalId"
    ] = predicted_animal_ids

    missing_animal_data[
        "PredictionConfidence"
    ] = prediction_confidence

    missing_animal_data[
        "AcceptedAnimalId"
    ] = np.where(
        missing_animal_data[
            "PredictionConfidence"
        ] >= CONFIDENCE_THRESHOLD,
        missing_animal_data[
            "PredictedAnimalId"
        ],
        np.nan
    )

    accepted_count = (
        missing_animal_data[
            "AcceptedAnimalId"
        ]
        .notna()
        .sum()
    )

    print(
        f"Accepted predictions at "
        f"{CONFIDENCE_THRESHOLD:.0%} confidence: "
        f"{accepted_count:,} / "
        f"{len(missing_animal_data):,}"
    )

    output_path = (
        PREDICTION_DATA_PATH.parent
        / "AnimalId_predictions.csv"
    )

    missing_animal_data.to_csv(
        output_path,
        index=False
    )

    print(
        f"Saved predictions to: "
        f"{output_path}"
    )

    display(
        missing_animal_data[
            [
                "EventDate",
                "LactationNumber",
                "DaysInMilk",
                "YieldSession",
                "PredictedAnimalId",
                "PredictionConfidence",
                "AcceptedAnimalId",
            ]
        ].head(20)
    )

else:
    print(
        "Final prediction is currently disabled. "
        "Set RUN_FINAL_PREDICTION = True after updating "
        "PREDICTION_DATA_PATH."
    )


## Interpretation and reporting notes

When reporting results, keep these quantities separate:

- **Coverage:** how many validation/test records belong to `AnimalId` classes already observed in the relevant training period.
- **Top-1 accuracy:** proportion of evaluable records assigned the exact correct `AnimalId`.
- **Top-5 accuracy:** proportion for which the true `AnimalId` is among the five highest-probability candidates.
- **Balanced accuracy / Macro F1:** give more equal weight to animals with fewer records.
- **Weighted F1:** reflects overall performance while accounting for class frequency.
- **Prediction confidence:** maximum class probability for each missing-ID prediction.

The `0.80` confidence threshold in the final section is only an initial operational threshold. It should ideally be calibrated using validation data before being treated as a final acceptance rule.
